# **Smart Water Systems: Query by Content**

## Learning goals
By the end of this notebook, students should be able to:

1. Explain what it means to **query a time series by content**.
2. Represent each day as a 24-hour profile.
3. Use **Euclidean distance** to measure similarity between daily profiles.
4. Find the days that are most similar to a selected query day.
5. Visualize the query profile and its closest matches.

We will use **hourly water use from ECJ** throughout the notebook.

In [ ]:
# import packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# **1. Load the ECJ data**

The CSV file contains hourly water, electricity, cooling, and heating data.

For this notebook, we will use only the **water** data.

In [ ]:
# Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')

# Define course folders
course_folder = '/content/drive/MyDrive/CE374U2_Fall2026'
data_folder = os.path.join(course_folder, 'data')

file_name = '03_ECJ.csv'
data_file = os.path.join(data_folder, file_name)

data_file

In [ ]:
df = pd.read_csv(data_file)
df.head()

### **Select and prepare the variables**

`pd.to_datetime()` converts the timestamp from text to a datetime object.

We will keep only the timestamp and water-use columns.

In [ ]:
# convert to datetime
df['DateTime'] = pd.to_datetime(df['DateTime'])

# keep only the variables needed in this notebook
df = df[['DateTime', 'Water']]

df.head()

In [ ]:
df.dtypes

### **Check the data**

Before comparing daily profiles, check the period of record and missing values.

In [ ]:
print('Start:', df['DateTime'].min())
print('End:', df['DateTime'].max())
print('Number of records:', len(df))
print('Missing water values:', df['Water'].isna().sum())

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(df['DateTime'], df['Water'])
plt.xlabel('Time')
plt.ylabel('Demand (Gallons)')
plt.title('Training and test periods')
plt.show()

In [ ]:
# lets check the data for basic statistics
df.describe()

# **2. What is query by content?**

Suppose we select one day and ask:

> **Which other days have the most similar water-use pattern?**

The selected day is the **query**.

Each day is represented by 24 hourly values:

$$Q = [q_0, q_1, q_2, \ldots, q_{23}]$$

Every other day is represented in the same way. We can then compare the query profile with each candidate day.

This is called **query by content** because we are searching for similar data patterns rather than searching by a date or label.

# **3. Create daily water-use profiles**

We first create two new variables:

- `date`: the calendar day
- `hour`: the hour of the day

Then we reshape the data so that:

- each **row** is one day
- each **column** is one hour

In [ ]:
df['date'] = df['DateTime'].dt.date
df['hour'] = df['DateTime'].dt.hour

df.head()

### **Reshape the time series into daily profiles**

The original data contains one row for each hour. To compare daily patterns, we will reshape the data so that:

- each **row** represents one day
- each **column** represents one hour of the day (0–23)
- each **value** is the water use at that hour

This converts each day into a 24-hour water-use profile:

$$
[W_0, W_1, W_2, \ldots, W_{23}]
$$

We can then compare these daily profiles to find days with similar water-use patterns.

The `pivot()` function performs this reshaping:

- `index='date'` → one row per day
- `columns='hour'` → one column per hour
- `values='Water'` → water use fills the table

In [ ]:
daily = df.pivot(index='date', columns='hour', values='Water')
daily.head()

A complete daily profile should have 24 hourly values. Any incomplete day is removed before we calculate distances.

In [ ]:
print('Number of days before removing incomplete days:', len(daily))

daily = daily.dropna()

print('Number of complete days:', len(daily))

# **4. Select a query day**

`daily` is a **pandas DataFrame**. Pandas is useful for organizing the data because it keeps the dates and hours attached to the water-use values.

For the calculations, we will represent each daily profile as a **NumPy array** containing 24 numbers.

First, we select a date from the index of `daily`. We then select the corresponding row and use `.to_numpy()` to convert the pandas row into a NumPy array.

The argument `dtype=float` ensures that all 24 values are stored as numeric floating-point values so that we can perform mathematical calculations with them.

Therefore:

- `daily` → pandas DataFrame containing all daily profiles
- `query_date` → date identifying the selected profile
- `query` → NumPy array containing the 24 hourly values for that day

In [ ]:
# let's choose first daily patter as our query
query_date = daily.index[💡]

query = daily.loc[💡].to_numpy(dtype=float)

In [ ]:
type(daily)

In [ ]:
type(query)

In [ ]:
query

In [ ]:
query.💡

In [ ]:
daily.💡

### Plot the query profile


In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(range(24), 💡, marker='o')

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Query profile: ' + str(query_date))
plt.xticks(range(0, 24, 3))
plt.grid(True)

plt.show()

# **5. Measure similarity using Euclidean distance**

For two daily profiles, \(Q\) and \(X\), the Euclidean distance is

$$d(Q,X)=\sqrt{\sum_{h=0}^{23}(Q_h-X_h)^2}$$

A **small distance** means the two daily profiles are similar.

A **large distance** means they are different.

### **Example: compare the query with one day**

First, calculate the distance to one candidate day step by step.

In [ ]:
daily.head()

In [ ]:
# get one example

example = daily.iloc[💡].to_numpy()
example


In [ ]:
# Calculate Euclidean distance
difference = query - example
squared_difference = difference ** 2
distance = np.sqrt(np.sum(squared_difference))


print('Euclidean distance:', round(distance, 2))

In [ ]:
# make a plot
plt.figure(figsize=(8, 4))

plt.plot(range(24), 💡, marker='o',label = 'query')
plt.plot(range(24), 💡, marker='s', label = 'daily pattern')

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Query profile: ' + str(query_date))
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

# **6. Calculate the distance to all daily profiles**

Now we compare the query with every daily profile in the dataset.

First, we create an array containing all daily profiles:

- `daily.to_numpy()` converts the daily profiles into a NumPy array.
- Each **row** represents one day and each **column** represents one hour.

If there are 365 days, `profiles` has a shape of:

$$
365 \times 24
$$

The query contains only one daily profile with 24 values:

$$
Q = [Q_0,Q_1,\ldots,Q_{23}]
$$

When we calculate:

`profiles - query`

NumPy automatically subtracts the 24 query values from **every row** in `profiles`. We do not need to write a loop. This is called **broadcasting**.

The result still has one row for each day and 24 columns representing the hourly differences.

Next:

- `** 2` squares each hourly difference.
- `sum(axis=1)` sums across the 24 hours for each day. **Easy way to remember:** axis=0 collapses the first dimension (rows), axis=1 collapses the second dimension (columns).
- `np.sqrt()` takes the square root of each sum.

Therefore, `distances` contains **one Euclidean distance for each day**:

$$
d(Q,X)=\sqrt{\sum_{h=0}^{23}(Q_h-X_h)^2}
$$

A smaller distance indicates a daily profile that is more similar to the query.

In [ ]:
# get all profiles
profiles = daily.to_numpy(dtype=float)
profiles

In [ ]:
# get distances for each hour and each row
squared_difference = (profiles - query) ** 2



In [ ]:
squared_difference.💡

In [ ]:
sum_squared_difference = squared_difference.sum(axis=1)


In [ ]:
sum_squared_difference.💡

In [ ]:
distances = np.sqrt(sum_squared_difference)
distances[:5]  # show the first five rows
# distances.shape

In [ ]:
#
# distances = np.sqrt(((profiles - query) ** 2).sum(axis=1))

# lets look at the first five
# distances[:5]

In [ ]:
# lets look at the distances

plt.plot(distances)

# **7. Find the closest matches**

`distances` contains one Euclidean distance for every daily profile.

To find the most similar profiles, we need to sort these distances from smallest to largest. We use `np.argsort()` to return the **indices** that would arrange the distances in increasing order.

The first result will be the query itself because its distance from itself is zero. We therefore skip the first result and keep the next 10.

Get the sorted indices

In [ ]:
sorted_indices = np.argsort(distances)

sorted_indices[:11]

Remove the query and keep the top 10

In [ ]:
match_indices = sorted_indices [💡]

# match_indices = sorted_indices[💡]

match_indices

The values in `match_indices` are the row positions of the 10 closest daily profiles.

We can use these same indices to retrieve both the corresponding distances and the dates.

In [ ]:
# get distances
match_distances = distances[💡]

match_distances

In [ ]:
# get the dates

match_dates = daily.index[💡]

match_dates

### Combine the results

The indices, dates, and distances refer to the same 10 matches. We can combine them into a table to make the results easier to interpret.

In [ ]:
matches = pd.DataFrame({
    'Index': match_indices,
    'Date': match_dates,
    'Distance': match_distances
})

matches

### **Plot the closest matches**

Distance gives us a numerical measure of similarity, but plotting the profiles helps us see what those similarities look like.

We will plot the query profile together with the five closest matches.

In [ ]:
n_closest = 💡

plt.figure(figsize=(10, 5))

# Plot the five closest matches
for i in match_indices[:💡]:
    plt.plot(range(24), profiles[💡], label=str(daily.index[💡]))

# Plot the query
plt.plot(range(24), 💡, linewidth=5, label='Query')

plt.xlabel('Hour')
plt.ylabel('Water use')
plt.title('Query Profile and Five Closest Matches')
plt.xticks(range(0, 24, 3))
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
import plotly.express as px

fig = px.line(
    df,
    x='DateTime',
    y='Water',
    title='Water Use and Closest Matching Profiles'
)

# Make the full time series light and thin
fig.update_traces(
    line=dict(color='lightgray', width=1),
    name='Full time series'
)
query_data = df[df['date'] == query_date]

fig.add_scatter(
    x=query_data['DateTime'],
    y=query_data['Water'],
    mode='lines',
    name='Query',
    line=dict(color='black', width=6)
)

for i, match_date in enumerate(match_dates):

    match_data = df[df['date'] == match_date]

    fig.add_scatter(
        x=match_data['DateTime'],
        y=match_data['Water'],
        mode='lines',
        name=str(match_date),
        line=dict(width=5)
    )
fig.update_layout(
    xaxis_title='Time',
    yaxis_title='Water Use'
)

fig.show()

In [ ]:
from google.colab import files

fig.write_html('query_matches_timeseries.html')
files.download('query_matches_timeseries.html')


### 💡 **Exercise: Explore Different Queries**
Try a different query by changing the:
1. starting time
2. query length
3. utility (electricity, cooling, or heating)

Run the query search again and compare the results.

# **8. Interpretation**

Answer the following questions:

1. Do the closest matches have a similar **shape**, similar **magnitude**, or both?
2. Are the closest matches near the query date, or do they occur at different times of the year?
3. What types of daily patterns might Euclidean distance consider similar?
4. What information does Euclidean distance **not** use when comparing the days?

## **Takeaway**

Query by content lets us search a time series for **similar patterns**.

In this notebook:

- one day was the query,
- each day was represented by 24 hourly values,
- Euclidean distance measured similarity,
- and the smallest distances identified the closest matches.

The same idea can be used with other variables, different time windows, and other distance measures.